# RAG Evaluation: Retrieval vs Generation

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/rag-evaluation)

Score RAG retrieval and generation quality independently with five metrics in one evaluate() call to pinpoint whether failures are at retrieval or generation.

## Install

Use a fresh Python 3.12 notebook kernel. The install cell uses the same Python interpreter as the notebook.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "agent-learning-kit==0.3.7"], check=True);

### Credentials

Set environment variables before starting the kernel, or enter the keys at the hidden prompts.

In [ ]:
import os
from getpass import getpass

for name in ('FI_API_KEY', 'FI_SECRET_KEY'):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")

## Set up a RAG test case

Define a realistic query, retrieved context chunks, and generated answer. This example simulates a Ridgeline Outfitters support assistant answering from its returns policy.

In [ ]:
query = "What is the refund policy and how long does processing take?"

chunks = [
    "Customers may request a full refund within 30 days of purchase. "
    "Refunds are processed within 5-7 business days after approval.",
    "To initiate a refund, contact support@example.com with your order number.",
    "Gift cards and promotional items are non-refundable.",
    "Ridgeline Outfitters was founded in 2015 and is headquartered in San Francisco.",
]
retrieved_context = "\n".join(chunks)

generated_answer = (
    "You can request a full refund within 30 days of purchase. "
    "Once approved, refunds are processed in 5-7 business days. "
    "To start, email support@example.com with your order number. "
    "Gift cards and promotional items cannot be refunded."
)

## Score retrieval quality

`evaluate()` takes an eval name (or a list of eval names), the fields that eval needs as keyword arguments, and a `model` (or `engine`) to run it on. Each of the five RAG metrics reads a different combination of `context`, `input`, and `output`:

In [ ]:
from fi.evals import evaluate

def checked(result):
    if result.status != "completed" or result.score is None or result.error:
        raise RuntimeError(f"{result.eval_name}: {result.error or result.status}")
    return result

# Context relevance: are the retrieved chunks relevant to the query?
relevance = evaluate(
    "context_relevance",
    context=retrieved_context,
    input=query,
    model="turing_flash",
)
print(f"Context Relevance: score={checked(relevance).score}, passed={relevance.passed}")
print(f"  Reason: {relevance.reason}\n")

# Chunk attribution: was the context chunk used in the response?
attribution = evaluate(
    "chunk_attribution",
    context=chunks[3],
    output=generated_answer,
    model="turing_flash",
)
print(f"Chunk Attribution (fourth chunk): score={checked(attribution).score}, passed={attribution.passed}")
print(f"  Reason: {attribution.reason}\n")

# Chunk utilization: how effectively does the response use the context?
utilization = evaluate(
    "chunk_utilization",
    context=retrieved_context,
    output=generated_answer,
    model="turing_flash",
)
print(f"Chunk Utilization: score={checked(utilization).score}, passed={utilization.passed}")
print(f"  Reason: {utilization.reason}\n")

## Score generation quality

These metrics evaluate whether the LLM fully addressed the query and produced factually accurate claims. To make the diagnostic concrete, score a deliberately wrong answer first, then fix it and rerun.

In [ ]:
from fi.evals import evaluate

# Completeness: does the response fully address the query?
completeness = evaluate(
    "completeness",
    input=query,
    output=generated_answer,
    model="turing_flash",
)
print(f"Completeness: score={checked(completeness).score}, passed={completeness.passed}")
print(f"  Reason: {completeness.reason}\n")

# Factual accuracy: introduce a claim the context doesn't support.
bad_answer = generated_answer.replace("5-7 business days", "24 hours")
bad_accuracy = evaluate(
    "factual_accuracy",
    input=query,
    output=bad_answer,
    context=retrieved_context,
    model="turing_flash",
)
print(f"Factual Accuracy (bad answer): score={checked(bad_accuracy).score}, passed={bad_accuracy.passed}")
print(f"  Reason: {bad_accuracy.reason}\n")

# Fix: match the processing time actually stated in the context, then rerun.
accuracy = evaluate(
    "factual_accuracy",
    input=query,
    output=generated_answer,
    context=retrieved_context,
    model="turing_flash",
)
print(f"Factual Accuracy (fixed): score={checked(accuracy).score}, passed={accuracy.passed}")
print(f"  Reason: {accuracy.reason}\n")

## Run all five metrics for a diagnostic overview

Pass a list of eval names to score several metrics in one call. The result comes back as an iterable `BatchResult`. This time `chunk_attribution` receives the combined context, not just the fourth chunk.

In [ ]:
from fi.evals import evaluate

diagnostic = evaluate(
    [
        "context_relevance",
        "chunk_attribution",
        "chunk_utilization",
        "completeness",
        "factual_accuracy",
    ],
    input=query,
    output=generated_answer,
    context=retrieved_context,
    model="turing_flash",
)

print("=== RAG Pipeline Diagnostic ===\n")
for result in diagnostic:
    checked(result)
    print(f"{result.eval_name:<22} score={result.score} passed={result.passed}")
    print(f"  Reason: {result.reason}\n")

## Interpret results: retrieval problem or generation problem?

Use the diagnostic output to decide where to focus your effort.

In [ ]:
# Build a name-keyed lookup, then gate on the real passed field.
scores = {result.eval_name: result for result in diagnostic}

retrieval_ok = scores["context_relevance"].passed
generation_ok = (
    all(scores[name].passed for name in ("chunk_attribution", "chunk_utilization", "completeness"))
    and scores["factual_accuracy"].score == 1.0
)

if not retrieval_ok:
    print("Action: improve retrieval. Check embeddings, re-ranking, or top-k settings.")
elif not generation_ok:
    print("Action: improve generation. Tune the prompt, lower temperature, or switch models.")
else:
    print("This refund case passed.")

Keep the evaluation IDs with their scores so you can look up these runs again.

In [ ]:
import json
from pathlib import Path

runs = [relevance, attribution, utilization, completeness, bad_accuracy, accuracy, *diagnostic]
report = [
    {"metric": r.eval_name, "eval_id": r.metadata["eval_id"], "score": r.score}
    for r in runs
]
Path("rag-results.json").write_text(json.dumps(report, indent=2))
print(f"Saved {len(report)} evaluations to rag-results.json")